In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
from pydantic import BaseModel, Field
import os
from langgraph.graph import StateGraph, START, END
api_key = os.getenv("GOOGLE_API_KEY")

if os.environ['GOOGLE_API_KEY']:
    print("Google api key is set")
else:
    raise ValueError("Gemini API key is not set")


llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", google_api_key=api_key, temperature=0.7)

Google api key is set


In [19]:
from langchain_core.tools import tool
from typing import TypedDict

@tool
def add(a: int, b:int) -> int :
    """Adding(+) numbers"""
    return a+b

@tool
def subtract(a:int, b:int) -> int:
    """Subtracting(-) numbers"""
    return a-b


@tool
def multiply(a:int, b:int) -> int:
    """Multiplying(*) numbers"""
    return a*b

@tool
def divide(a:int, b:int) -> int:
    """Dividing (/) numbers"""
    return a/b

In [20]:
tools = [add, subtract, multiply, divide]

llm_with_tools = llm.bind_tools(tools)
result = llm_with_tools.invoke("what is 25*782 then divided by 3")
print(result)


c:\FAHEEM\My_Programs\AI Agents\practice_AI_Agents\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


content=[] additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 782, "a": 25}'}, '__gemini_function_call_thought_signatures__': {'3JvNU2w8': 'EpkCCpYCARFNMg/HGeDEna55uJwMda4MGgFx0IfvY/MgGl4ldgzAh+B2sBljWcpu6I6KCdKsPCqY4glOI8j3C8kfGE3OS3cp8ZHz7a9OOUA2O8xU4N9EprSZzutUPWHI+m6DzRAnKZ8qLdPsOWDNesB/8TmrLjc5cdctw01lWSYMVoWB6amu+DxcpPu23CuPOoOyouudhNNJaoogHIxJjlkLumOHiqc3o1pHQc3DQpd1uCZbSY0KVQ/BoyT7wRx3hiO5eaaHpzBtMc1wdNAAuMM1eju7CpqAHGhVon4CkBiuavC6dpbah0/uTU00hNmYqOX+nK6kW5GQ0jz6T0E7mFgEK8JOnOg0KumraAgbvdJrxXI+raobaGZcaH8='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019fa474-3f79-7bd1-8de1-b89e7cd2f97c-0' tool_calls=[{'name': 'multiply', 'args': {'b': 782, 'a': 25}, 'id': '3JvNU2w8', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 219, 'output_tokens': 99, 'total_tokens': 318, 'input_token_details': {'cache_read': 0}, 'output_token_detai

In [21]:
from typing import TypedDict,Annotated, List
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, ToolMessage, HumanMessage

class graph_schema(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

In [22]:
def llm_function(state: graph_schema) -> graph_schema:
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    state['messages'] = messages + [response]
    return state

def tool_node(state: graph_schema) -> graph_schema:
    message = state['messages']
    tool_by_name = {tool.name: tool for tool in tools}
    tool_results = []

    for tool_calls in message[-1].tool_calls:
        tool = tool_by_name[tool_calls["name"]]

        observation = tool.invoke(tool_calls["args"])

        tool_results.append(ToolMessage(content=str(observation), tool_call_id=tool_calls["id"]))

    state['messages'] = message + tool_results
    return state

def if_tool_call(state: graph_schema) -> str:
    last_message = state['messages'][-1]
    if last_message.tool_calls:
        return "tool_node"
    else:
        return "end"

In [23]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(graph_schema)

graph.add_node("initial_node", llm_function)
graph.add_node("tool_node", tool_node)

graph.add_edge(START, "initial_node")
graph.add_conditional_edges("initial_node", if_tool_call, {"tool_node": "tool_node", "end": END})
graph.add_edge("tool_node", "initial_node")
graph.add_edge("initial_node", END)

calculator_graph = graph.compile()



In [24]:
result = calculator_graph.invoke({"messages":[HumanMessage(content="what is 6/3 then * 782 then divided by 2")]})
result


c:\FAHEEM\My_Programs\AI Agents\practice_AI_Agents\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\FAHEEM\My_Programs\AI Agents\practice_AI_Agents\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\FAHEEM\My_Programs\AI Agents\practice_AI_Agents\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\FAHEEM\My_Programs\AI Agents\practice_AI_Agents\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flas

{'messages': [HumanMessage(content='what is 6/3 then * 782 then divided by 2', additional_kwargs={}, response_metadata={}, id='5071350c-3752-4f72-9a63-6e51a316facd'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'divide', 'arguments': '{"b": 3, "a": 6}'}, '__gemini_function_call_thought_signatures__': {'z54VUI65': 'Ev8CCvwCARFNMg/bElVih493nMZvOcc3RnYuyfF5uLHgwaJ0dcSMHhvMLlk5+Gk3j3nVDm29V/KJA95T8sQblj09caYfXCFJcaWserdThCfkhDiWoZ07X/PXWYhrR10CHQdkGkXQ01sztktB1Ddy8kpiblBl5AQUNMlBriP6vhS9DhwRe2b0VA087SoilYj8JW5ANkLkdWBRO9Vi5wCP4sDUuXFfVT90l+WGlpBTLZLCa8tlpKUd49eLSb++q6pEA+tQ/nrSOrJD3Ld8k4uEA5wwyzUmmQJ/hKrcDJNCO4XnZa25fau1qrIgEQkm/oFu3yM8ILPZQdH6abH53SB+G21ytIW8EEpsdYQhGbV+3RfM6VUiRHqxvc8SaPTjl+rzQq/EWt1cbKFbY97jIwN00B3YFYVnNaKpWQxT7ajcNkdz2EBSQ6WM3MStpYHEB0BTZr6h8miWR7y7sXLqlwoXLUdyI1NSlgYNU3MUxE2BfICzS1iDzbvcUMHkbdHSY3BuEX8='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id=